In [1]:
import sys
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, Subset
from torchvision.utils import make_grid
import torch.nn.functional as F


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random


import models as models
import train_helper as train_helper
import utils as utils
import data_helper as data_helper

vae_path = "/home/benjiy/repo/Verified-Synthetic-Data/MNIST"
sys.path.append(vae_path)

In [2]:
# Set up device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_seed = 0
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)

model_saved_path = os.path.join(os.getcwd(),"model_saved")
data_saved_path = os.path.join(os.getcwd(),"data_saved")
results_saved_path = os.path.join(os.getcwd(),"results_saved")




## Prepare data ##


In [3]:


full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
full_digit_indices = utils.create_balanced_subset_indices(full_dataset,seed=base_seed)

#train_dataset_5000 = Subset(full_dataset, range(5000))


## Load/train small model

In [5]:
init_size = 5000

init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
train_loader_5000 = DataLoader(init_dataset, batch_size=128, shuffle=True)
model_5000 = models.CVAE(input_dim=784, label_dim=10,latent_dim=20,name="cvae_conv_5000",arch="conv").to(device)
#train_logs = training.train_model(model, train_loader_5000, device, epochs=200, lr=1e-3, patience=5)
train_helper.train_model_with_validation(model=model_5000, train_loader=train_loader_5000, val_loader=test_loader, device=device, epochs=200, lr=1e-3, patience=5)
# utils.save_model(model_5000, "cvae_real_5000_new", model_saved_path)

# model_5000 = utils.load_model("cvae_real_5000_new", model_saved_path , device, (784, 10, 20))


Epoch [1/200] completed. Train Avg Loss: 323.4146, Val Avg Loss: 208.9439
Train Summary: recon: 303.5401, kld: 19.8745, Val Summary: recon: 198.3694, kld: 10.5745
Epoch [2/200] completed. Train Avg Loss: 189.7755, Val Avg Loss: 168.0777
Train Summary: recon: 178.3544, kld: 11.4211, Val Summary: recon: 153.7998, kld: 14.2778
Epoch [3/200] completed. Train Avg Loss: 160.5661, Val Avg Loss: 150.8710
Train Summary: recon: 144.4074, kld: 16.1587, Val Summary: recon: 132.1784, kld: 18.6926
Epoch [4/200] completed. Train Avg Loss: 146.5065, Val Avg Loss: 139.1821
Train Summary: recon: 127.2772, kld: 19.2293, Val Summary: recon: 117.8068, kld: 21.3753
Epoch [5/200] completed. Train Avg Loss: 136.8254, Val Avg Loss: 131.2739
Train Summary: recon: 115.6620, kld: 21.1634, Val Summary: recon: 109.5557, kld: 21.7181
Epoch [6/200] completed. Train Avg Loss: 130.3900, Val Avg Loss: 125.9227
Train Summary: recon: 107.7274, kld: 22.6626, Val Summary: recon: 101.9571, kld: 23.9656
Epoch [7/200] complete

{'train_losses': [323.41459729003907,
  189.77548278808592,
  160.56612163085939,
  146.50653588867186,
  136.82536228027342,
  130.3899756713867,
  125.75780646972656,
  122.6357905029297,
  120.335324609375,
  118.39895427246094,
  117.09924663085937,
  115.62886439208984,
  114.5365952392578,
  113.50049572753906,
  112.44307725830078,
  111.82226403808593,
  111.1158917602539,
  110.2437560546875,
  109.69084926757813,
  109.06256625976563,
  108.74643842773438,
  107.99079396972657,
  107.78415501708984,
  107.34986733398438,
  106.92305997314453,
  106.44223859863281,
  106.34655322265625,
  105.9908609008789,
  105.57559868164063,
  105.29108435058593,
  105.16334987792969,
  104.83709915771485,
  104.50859428710937,
  104.33821651611328,
  104.27197611083984,
  104.01134313964843,
  103.48343681640625,
  103.51977092285156,
  103.41899265136719,
  103.33765551757813,
  102.98905755615235,
  102.65680948486329,
  102.65421026611328,
  102.5147114013672,
  102.40113439941406,
  1

In [ ]:
#utils.plot_samples_per_digit(8, model_5000)

### Load/Train Full model

In [ ]:

 
train_loader_60000 = DataLoader(full_dataset, batch_size=128, shuffle=True)
model_60000 = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
train_logs_60000 = train_helper.train_model_with_validation(model_60000, train_loader_60000, test_loader, device, epochs=200, lr=1e-3, patience=5)
utils.save_model(model_60000, "cvae_real_60000_new", model_saved_path)

#model_60000 = utils.load_model("cvae_real_60000_new", model_saved_path , device, (784, 10, 20))


## Train Discriminator


In [ ]:
# Train Discriminator
discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, model_5000, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
discmodel_5000 = models.SyntheticDiscriminator(input_dim=784)
disc_logs_5000 = train_helper.train_model(model=discmodel_5000, train_loader=disc_loader, device=device, epochs=50, lr=1e-3, patience=5)
utils.save_model(discmodel_5000, "disc_5000_sample60000_new", model_saved_path)

# discmodel_5000 = utils.load_model("disc_5000_sample60000_new", model_saved_path , device, (784,))


## Retrain with synthetic data

### generate synthetic data

In [ ]:
#data_helper.generate_images_with_filtering(model_5000, data_saved_path, "cvae_real_5000_new", total_samples=240000, batch_size=60000, discriminator=discmodel_5000, selection_percentile=1, per_digit_filtering=True)

data_helper.generate_balanced_images_with_filtering(model=model_5000, save_directory=data_saved_path, model_name="cvae_real_5000_new",
 total_samples=300000, batch_size=60000, discriminator=discmodel_5000, selection_threshold=0.5, use_quantile_filtering=False)

### check synthetic data

In [ ]:
utils.display_samples_from_pt_file(5,os.path.join(data_saved_path,"cvae_real_5000_new","cvae_real_5000_new_60000_t0.5_b0.pt"))

### retrain with synthetic data

In [ ]:
synthetic_data_load_path = os.path.join(data_saved_path,"cvae_real_5000_new")
synthetic_train_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
utils.verify_balance(synthetic_train_loader.dataset)

In [ ]:

synthetic_train_logs = train_helper.train_model_with_validation(synthetic_model, synthetic_train_loader, test_loader, device, epochs=200, lr=1e-3, patience=5,verbose=True)
#synthetic_train_logs = train_helper.train_model(synthetic_model, synthetic_train_loader, device, epochs=200, lr=1e-3, patience=5,verbose=True)

In [ ]:
print(train_helper.calculate_validation_loss(model_5000, test_loader, device))
# print(train_helper.calculate_validation_loss(model_60000, test_loader, device))
print(train_helper.calculate_validation_loss(synthetic_model, test_loader, device))


## Train model with different sizes

In [ ]:
model_sizes = [1000, 2000, 5000, 10000, 20000, 30000, 40000, 50000, 60000]
all_logs = []
all_models = []
test_error = {"val_loss":[], "kl":[], "recon":[]}
for model_size in model_sizes:
    balanced_subset = utils.get_balanced_subset(full_digit_indices, model_size)
    train_dataset_balanced = Subset(full_dataset, balanced_subset)
    train_loader_balanced = DataLoader(train_dataset_balanced, batch_size=128, shuffle=True)
    model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_{model_size}_balanced").to(device)
    train_logs = train_helper.train_model_with_validation(model, train_loader_balanced, test_loader, device, epochs=200, lr=1e-3, patience=5,verbose=False)
    utils.save_model(model=model, model_name=model.name, path=model_saved_path)
    val_loss, val_kl, val_recon = train_helper.calculate_validation_loss(model, test_loader, device)
    test_error["val_loss"].append(val_loss)
    test_error["kl"].append(val_kl)
    test_error["recon"].append(val_recon)
    all_logs.append(train_logs)
    all_models.append(model)
    print(f"Model size: {model_size}, Val Loss: {val_loss}, Val KL: {val_kl}, Val Recon: {val_recon}")



In [ ]:
len(all_models)
test_error = {"val_loss":[], "kl":[], "recon":[]}
for model in all_models:
    val_loss, val_kl, val_recon = train_helper.calculate_validation_loss(model, test_loader, device)
    test_error["val_loss"].append(val_loss)
    test_error["kl"].append(val_kl)
    test_error["recon"].append(val_recon)    

test_error['model_size'] = model_sizes
res_table = pd.DataFrame.from_dict(test_error, orient='columns')
print(res_table)
res_table.to_csv(os.path.join(results_saved_path, 'diff_modelsize_results.csv'), index=False)


## Repeat single step retraining process with different seeds

In [ ]:
model_size = 5000
synthetic_size = 60000
test_results = {"original_loss":[], "original_kl":[], "original_recon":[], "synthetic_loss":[], "synthetic_kl":[], "synthetic_recon":[]}
orig_subset = utils.get_balanced_subset(full_digit_indices, model_size)
orig_dataset = Subset(full_dataset, orig_subset)
orig_loader = DataLoader(orig_dataset, batch_size=128, shuffle=True)
num_experiments = 10
for exp_num in range(num_experiments):
    print(f"\n=== Experiment {exp_num + 1}/{num_experiments} ===")

    exp_seed = base_seed + exp_num
    torch.manual_seed(exp_seed)
    torch.cuda.manual_seed_all(exp_seed)
    np.random.seed(exp_seed)
    random.seed(exp_seed)

    # Train Original Model
    orig_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_{model_size}_repeat_{exp_seed}").to(device)
    train_helper.train_model(orig_model, orig_loader, device, epochs=200, lr=1e-3, patience=5)    
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(orig_model, test_loader, device)
    test_results["original_loss"].append(val_loss)
    test_results["original_kl"].append(val_kl)
    test_results["original_recon"].append(val_recon)

    # Train Discriminator
    discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, orig_model, device)
    disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
    disc_model = models.SyntheticDiscriminator(input_dim=784)    
    train_helper.train_model(disc_model, disc_loader, device, epochs=80, lr=1e-3, patience=5)

    # Generate Synthetic Data
    data_helper.generate_images_with_filtering(orig_model, data_saved_path, orig_model.name, 
    total_samples=synthetic_size, batch_size=60000, discriminator=disc_model, selection_percentile=1, per_digit_filtering=True, verbose=False)

    # Train Synthetic Model
    synthetic_data_load_path = os.path.join(data_saved_path,orig_model.name)
    synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
    synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20).to(device)
    train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5)
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
    test_results["synthetic_loss"].append(val_loss)
    test_results["synthetic_kl"].append(val_kl)
    test_results["synthetic_recon"].append(val_recon)

res_table = pd.DataFrame.from_dict(test_results, orient='columns')


In [ ]:
summary_stats = {
    'Metric': ['Loss', 'KL Divergence', 'Reconstruction'],
    'Original_Mean': [
        np.mean(test_results["original_loss"]),
        np.mean(test_results["original_kl"]),
        np.mean(test_results["original_recon"])
    ],
    'Original_Std': [
        np.std(test_results["original_loss"]),
        np.std(test_results["original_kl"]),
        np.std(test_results["original_recon"])
    ],
    'Synthetic_Mean': [
        np.mean(test_results["synthetic_loss"]),
        np.mean(test_results["synthetic_kl"]),
        np.mean(test_results["synthetic_recon"])
    ],
    'Synthetic_Std': [
        np.std(test_results["synthetic_loss"]),
        np.std(test_results["synthetic_kl"]),
        np.std(test_results["synthetic_recon"])
    ],
    'Improvement_Mean': [
        np.mean([o - s for o, s in zip(test_results["original_loss"], test_results["synthetic_loss"])]),
        np.mean([o - s for o, s in zip(test_results["original_kl"], test_results["synthetic_kl"])]),
        np.mean([o - s for o, s in zip(test_results["original_recon"], test_results["synthetic_recon"])])
    ],
    'Improvement_Std': [
        np.std([o - s for o, s in zip(test_results["original_loss"], test_results["synthetic_loss"])]),
        np.std([o - s for o, s in zip(test_results["original_kl"], test_results["synthetic_kl"])]),
        np.std([o - s for o, s in zip(test_results["original_recon"], test_results["synthetic_recon"])])
    ]
}

summary_df = pd.DataFrame(summary_stats)
print(summary_df)

## Iterative Re-Training

In [ ]:
init_size = 30000

test_results = {"val_loss":[], "val_recon":[], "val_kl":[]}
init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
init_loader = DataLoader(init_dataset, batch_size=128, shuffle=True)


this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_iter_{init_size}").to(device)
train_helper.train_model(this_model, init_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(this_model, test_loader, device)
test_results["val_loss"].append(val_loss)
test_results["val_recon"].append(val_recon)
test_results["val_kl"].append(val_kl)
utils.save_model(this_model, this_model.name, model_saved_path)

synthetic_size_schedule = [60000, 100000, 150000, 200000, 300000]
for synthetic_size in synthetic_size_schedule:
    print(f"\n=== Synthetic Size: {synthetic_size} ===")
    # Train Discriminator
    discriminator_dataset = data_helper.prepare_discriminator_dataset(full_dataset, this_model, device)
    disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)

    discriminator_test_dataset = data_helper.prepare_discriminator_dataset(test_dataset, this_model, device)
    disc_test_loader = DataLoader(discriminator_test_dataset, batch_size=128, shuffle=True)

    disc_model = models.SyntheticDiscriminator(input_dim=784)    
    train_helper.train_model_with_validation(disc_model, disc_loader, disc_test_loader, device, epochs=80, lr=1e-3, patience=5, verbose=False)

    # Generate Synthetic Data
    data_helper.generate_balanced_images_with_filtering(model=this_model, save_directory=data_saved_path, model_name=this_model.get_name(), 
    total_samples=synthetic_size, discriminator=disc_model, selection_threshold=0.5, verbose=False)

    # Train Synthetic Model
    synthetic_data_load_path = os.path.join(data_saved_path,this_model.name)
    synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
    synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20, name=f"cvae_iter_{synthetic_size}").to(device)
    train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
    test_results["val_loss"].append(val_loss)
    test_results["val_recon"].append(val_recon)
    test_results["val_kl"].append(val_kl)

    print(f"Synthetic Size: {synthetic_size} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")

    utils.save_model(synthetic_model, synthetic_model.get_name(), model_saved_path)
    this_model = synthetic_model

    del synthetic_loader
    del disc_model
    del discriminator_dataset
    del disc_loader
    

res_table = pd.DataFrame.from_dict(test_results, orient='columns')


In [ ]:
res_table

## Training results check

In [ ]:
utils.plot_samples_per_digit(5,synthetic_model)

In [ ]:
utils.plot_samples_per_digit(5,model_5000)


# check discriminator improvement

In [7]:
disc_dataset_base = data_helper.prepare_discriminator_dataset(full_dataset, model_5000, device)
disc_loader_base = DataLoader(disc_dataset_base, batch_size=128, shuffle=True)
disc_test_dataset_base = data_helper.prepare_discriminator_dataset(test_dataset, model_5000, device)
disc_test_loader_base = DataLoader(disc_test_dataset_base, batch_size=128, shuffle=True)

discmodel_5000_base = models.SyntheticDiscriminator(input_dim=784)
train_helper.train_model_with_validation(model=discmodel_5000_base, train_loader=disc_loader_base, val_loader=disc_test_loader_base, device=device, epochs=100, lr=1e-3, patience=5)


Epoch [1/100] completed. Train Avg Loss: 0.5231, Val Avg Loss: 0.4305
Train Summary: accuracy: 0.7254, Val Summary: accuracy: 0.7934
Epoch [2/100] completed. Train Avg Loss: 0.3475, Val Avg Loss: 0.3179
Train Summary: accuracy: 0.8452, Val Summary: accuracy: 0.8624
Epoch [3/100] completed. Train Avg Loss: 0.2541, Val Avg Loss: 0.2663
Train Summary: accuracy: 0.8931, Val Summary: accuracy: 0.8941
Epoch [4/100] completed. Train Avg Loss: 0.1882, Val Avg Loss: 0.2591
Train Summary: accuracy: 0.9233, Val Summary: accuracy: 0.8935
Epoch [5/100] completed. Train Avg Loss: 0.1462, Val Avg Loss: 0.2625
Train Summary: accuracy: 0.9414, Val Summary: accuracy: 0.9056
Epoch [6/100] completed. Train Avg Loss: 0.1252, Val Avg Loss: 0.2344
Train Summary: accuracy: 0.9502, Val Summary: accuracy: 0.9154
Epoch [7/100] completed. Train Avg Loss: 0.0978, Val Avg Loss: 0.2470
Train Summary: accuracy: 0.9613, Val Summary: accuracy: 0.9041
Epoch [8/100] completed. Train Avg Loss: 0.0854, Val Avg Loss: 0.2046

{'train_losses': [0.5231122101942698,
  0.34749774031639097,
  0.2540977737824122,
  0.18816317228476206,
  0.1462356472571691,
  0.12518779440522193,
  0.09775916281342506,
  0.0854375437895457,
  0.07392834195892016,
  0.06707231060465177,
  0.06313411592543125,
  0.055293953809390464,
  0.05191291288783153,
  0.047248559035360814,
  0.044635494716713825,
  0.041325884493192036,
  0.04149920882980029,
  0.039619911460826794,
  0.03380645413659513,
  0.03517593579081198,
  0.030891142703654867,
  0.030839802233005562,
  0.028129415879460672,
  0.02972303612108032,
  0.028430046760477126,
  0.02591642292526861,
  0.025914978956182798,
  0.024992579163610935,
  0.02461903607727339,
  0.02512111391875272,
  0.02288798558463653,
  0.020228278474913288,
  0.021411781687041123,
  0.01937352216138194,
  0.021473508909655114,
  0.017339149603806437,
  0.019961751803724715,
  0.016877326412623128,
  0.020407621318055316,
  0.016707748567825184,
  0.015744678481528534,
  0.015957097137183882,
 

In [8]:
disc_dataset_label = data_helper.prepare_discriminator_dataset_with_labels(full_dataset, model_5000, device)
disc_loader_label = DataLoader(disc_dataset_label, batch_size=128, shuffle=True)
disc_test_dataset_label = data_helper.prepare_discriminator_dataset_with_labels(test_dataset, model_5000, device)
disc_test_loader_label = DataLoader(disc_test_dataset_label, batch_size=128, shuffle=True)




In [ ]:
discmodel_5000_mlp = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_mlp", arch="mlp")
train_helper.train_model_with_validation(model=discmodel_5000_mlp, train_loader=disc_loader_label, val_loader=disc_test_loader_label, device=device, epochs=200, lr=1e-3, patience=5)

Epoch [1/100] completed. Train Avg Loss: 0.6377, Val Avg Loss: 0.6161
Train Summary: accuracy: 0.6492, Val Summary: accuracy: 0.6884
Epoch [2/100] completed. Train Avg Loss: 0.6052, Val Avg Loss: 0.5971
Train Summary: accuracy: 0.7067, Val Summary: accuracy: 0.7259
Epoch [3/100] completed. Train Avg Loss: 0.5954, Val Avg Loss: 0.5956
Train Summary: accuracy: 0.7234, Val Summary: accuracy: 0.7229
Epoch [4/100] completed. Train Avg Loss: 0.5898, Val Avg Loss: 0.5895
Train Summary: accuracy: 0.7327, Val Summary: accuracy: 0.7424
Epoch [5/100] completed. Train Avg Loss: 0.5851, Val Avg Loss: 0.5911
Train Summary: accuracy: 0.7409, Val Summary: accuracy: 0.7215
Epoch [6/100] completed. Train Avg Loss: 0.5812, Val Avg Loss: 0.5808
Train Summary: accuracy: 0.7493, Val Summary: accuracy: 0.7578
Epoch [7/100] completed. Train Avg Loss: 0.5789, Val Avg Loss: 0.5810
Train Summary: accuracy: 0.7527, Val Summary: accuracy: 0.7517
Epoch [8/100] completed. Train Avg Loss: 0.5759, Val Avg Loss: 0.5769

{'train_losses': [0.637673877398173,
  0.6051528254191081,
  0.5953675633748372,
  0.5897987712860108,
  0.5851379173278809,
  0.5812318755785624,
  0.5789119660059611,
  0.5758983870824178,
  0.5739951992034912,
  0.571770042069753,
  0.5699629467010499,
  0.5676329905192057,
  0.5660771678288777,
  0.5643046572367351,
  0.5626594456354777,
  0.561394852288564,
  0.5598012254079183,
  0.5587421641349792,
  0.5573504923502605,
  0.556326572672526,
  0.5554765527089437,
  0.5545707535107931,
  0.5530504446983338,
  0.5527125463167827,
  0.5515349954922995,
  0.5507899225234986,
  0.5498096659978231,
  0.5487044870058696,
  0.5486053860028585,
  0.548089582379659,
  0.5470109131177266,
  0.5464772514343261,
  0.545784468905131,
  0.5451701866149903,
  0.5447291333198547,
  0.5440738862673442,
  0.5437719713846842,
  0.542938305123647,
  0.5431105590820312,
  0.5423565965970357,
  0.5417434483528137,
  0.5410540102005005,
  0.5408985397656758,
  0.54037358973821,
  0.5402005870183308,
  0

In [10]:
discmodel_5000_conv = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_conv", arch="conv")
train_helper.train_model_with_validation(model=discmodel_5000_conv, train_loader=disc_loader_label, val_loader=disc_test_loader_label, device=device, epochs=200, lr=1e-3, patience=5)

Epoch [1/200] completed. Train Avg Loss: 0.5600, Val Avg Loss: 0.3771
Train Summary: accuracy: 0.7052, Val Summary: accuracy: 0.8780
Epoch [2/200] completed. Train Avg Loss: 0.3649, Val Avg Loss: 0.3526
Train Summary: accuracy: 0.8976, Val Summary: accuracy: 0.9227
Epoch [3/200] completed. Train Avg Loss: 0.3284, Val Avg Loss: 0.3066
Train Summary: accuracy: 0.9162, Val Summary: accuracy: 0.9356
Epoch [4/200] completed. Train Avg Loss: 0.3132, Val Avg Loss: 0.2996
Train Summary: accuracy: 0.9243, Val Summary: accuracy: 0.9499
Epoch [5/200] completed. Train Avg Loss: 0.3037, Val Avg Loss: 0.2983
Train Summary: accuracy: 0.9291, Val Summary: accuracy: 0.9405
Epoch [6/200] completed. Train Avg Loss: 0.2985, Val Avg Loss: 0.2813
Train Summary: accuracy: 0.9315, Val Summary: accuracy: 0.9473
Epoch [7/200] completed. Train Avg Loss: 0.2906, Val Avg Loss: 0.3201
Train Summary: accuracy: 0.9369, Val Summary: accuracy: 0.9508
Epoch [8/200] completed. Train Avg Loss: 0.2892, Val Avg Loss: 0.2692

{'train_losses': [0.5600227400461832,
  0.364851336479187,
  0.328425061750412,
  0.31321594398816427,
  0.3037407353878021,
  0.29848905288378397,
  0.2905975032329559,
  0.28921438008944195,
  0.28407392988204955,
  0.2811470681667328,
  0.2778369042396545,
  0.2761077007929484,
  0.2724611138661702,
  0.27101948068936665,
  0.26873033396402995,
  0.2661112324873606,
  0.2657323954264323,
  0.26493239795366924,
  0.2632438482046127,
  0.2619634587287903,
  0.25993874400456746,
  0.26102145764033,
  0.2587105771064758,
  0.25836671854654947,
  0.2554482891956965,
  0.2554122252782186,
  0.2545477799574534,
  0.2530286196072896,
  0.25239348309834797,
  0.25205758571624753,
  0.2516371035416921,
  0.25029828582604724,
  0.24929441610177358,
  0.24795061948299407,
  0.24795768526395162,
  0.24763614616394042,
  0.2460823558807373,
  0.24638221559524537,
  0.2452841115474701,
  0.24386353022257487,
  0.24411998824278514,
  0.24338645186424254,
  0.2443370492299398,
  0.24241872398058573,

## Try retraining with better discriminator

In [17]:
all_models = []

In [20]:
all_models

[CVAE(name=conv_conv_real_5000, latent_dim=20),
 CVAE(name=cvae_conv_disc_conv_q0.2_200000_on_real_5000, latent_dim=20)]

In [ ]:
import shutil

test_results = {"val_loss":[], "val_recon":[], "val_kl":[], "model_name":[]}

# init_size = 5000
# init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
# init_dataset = Subset(full_dataset, init_subset)
# init_loader = DataLoader(full_dataset, batch_size=128, shuffle=True)


# this_model = models.CVAE(input_dim=784, label_dim=10, latent_dim=20, name=f"cvae_conv_real_full",arch="conv").to(device)
# train_helper.train_model(this_model, init_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)

# this_model = utils.load_model(model_name="cvae_conv_real_5000", path=model_saved_path, input_device=device, model_args=(784, 10, 20, "conv_conv_real_5000", "conv"))
this_model = all_models[-1]
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(this_model, test_loader, device)
test_results["val_loss"].append(val_loss)
test_results["val_recon"].append(val_recon)
test_results["val_kl"].append(val_kl)
test_results["model_name"].append(this_model.name)


print(f"Model: {this_model.get_name()} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")
# all_models.append(this_model)

discriminator_dataset = data_helper.prepare_discriminator_dataset_with_labels(full_dataset, this_model, device)
disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)
discriminator_test_dataset = data_helper.prepare_discriminator_dataset_with_labels(test_dataset, this_model, device)
disc_test_loader = DataLoader(discriminator_test_dataset, batch_size=128, shuffle=True)
disc_model = models.ConditionalDiscriminator(input_dim=784, name="disc_5000_conv", arch="conv")
#train_helper.train_model(model=disc_model, train_loader=disc_loader, device=device, epochs=200, lr=1e-3, patience=5, verbose=True)
train_logs = train_helper.train_model_with_validation(model=disc_model, train_loader=disc_loader, val_loader=disc_test_loader, device=device, epochs=200, lr=1e-3, patience=5, verbose=True)




Model: cvae_conv_disc_conv_q0.2_200000_on_real_5000 - Val Loss: 98.5887 - Val KL: 22.5896 - Val Recon: 75.9991
Epoch [1/200] completed. Train Avg Loss: 0.6076, Val Avg Loss: 0.5091
Train Summary: accuracy: 0.6608, Val Summary: accuracy: 0.7962
Epoch [2/200] completed. Train Avg Loss: 0.4651, Val Avg Loss: 0.4226
Train Summary: accuracy: 0.8283, Val Summary: accuracy: 0.8897
Epoch [3/200] completed. Train Avg Loss: 0.4106, Val Avg Loss: 0.4375
Train Summary: accuracy: 0.8716, Val Summary: accuracy: 0.7949
Epoch [4/200] completed. Train Avg Loss: 0.3831, Val Avg Loss: 0.3568
Train Summary: accuracy: 0.8906, Val Summary: accuracy: 0.9053
Epoch [5/200] completed. Train Avg Loss: 0.3642, Val Avg Loss: 0.3470
Train Summary: accuracy: 0.9028, Val Summary: accuracy: 0.9222
Epoch [6/200] completed. Train Avg Loss: 0.3523, Val Avg Loss: 0.3551
Train Summary: accuracy: 0.9111, Val Summary: accuracy: 0.8856
Epoch [7/200] completed. Train Avg Loss: 0.3427, Val Avg Loss: 0.3436
Train Summary: accura

{'train_losses': [0.6075885115305583,
  0.4650663490931193,
  0.4106422578175863,
  0.38307904669443765,
  0.36415155450503034,
  0.35230426440238954,
  0.3427195782661438,
  0.33708088477452597,
  0.3304639369328817,
  0.3281658098856608,
  0.32389829672177634,
  0.3193101871172587,
  0.3161919257005056,
  0.3149074315706889,
  0.311885621992747,
  0.31014232374827067,
  0.3090874711672465,
  0.3057156397660573,
  0.30670373301506043,
  0.3026385585308075,
  0.3004848272959391,
  0.2993290666898092,
  0.29832413727442425,
  0.296748345263799,
  0.29523819829622905,
  0.29436168508529664,
  0.29376244099934895,
  0.29158821840286253,
  0.2911199810187022,
  0.29022022013664245,
  0.2887079831123352,
  0.28826114652951557,
  0.2884449505329132,
  0.2856243682384491,
  0.28715956953366595,
  0.2842678491910299,
  0.2835559206167857,
  0.2839262376944224,
  0.28134604495366416,
  0.28155786860783893,
  0.28070731910069785,
  0.28163981097539265,
  0.28026767180760703,
  0.2788004865328471

In [ ]:

# utils.save_model(this_model, this_model.get_name(), model_saved_path)

for threshold in [0.2,0.8,0.1,0.5]:
    for synthetic_size in [200_000,500_000,1_000_000]:
        # Generate Synthetic Data
        model_name = f'cvae_conv_disc_conv_q{threshold}_{synthetic_size}_on_conv_q0.2_200000'
        synthetic_data_load_path = os.path.join(data_saved_path,model_name)
        data_helper.generate_balanced_images_with_filtering(model=this_model, save_directory=synthetic_data_load_path, 
        total_samples=synthetic_size, discriminator=disc_model, selection_threshold=threshold, verbose=False, use_quantile_filtering=True)

        # Train Synthetic Model

        synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
        synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20, name=model_name,arch="conv").to(device)
        train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
        val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(synthetic_model, test_loader, device)
        test_results["val_loss"].append(val_loss)
        test_results["val_recon"].append(val_recon)
        test_results["val_kl"].append(val_kl)
        test_results["model_name"].append(synthetic_model.get_name())

        print(f"Model Name: {model_name} - Val Loss: {val_loss:.4f} - Val KL: {val_kl:.4f} - Val Recon: {val_recon:.4f}")

        utils.save_model(synthetic_model, synthetic_model.get_name(), model_saved_path)
        all_models.append(synthetic_model)

        del synthetic_loader

res_table = pd.DataFrame.from_dict(test_results, orient='columns')

Model Name: cvae_conv_disc_conv_q0.2_200000_on_conv_q0.2_200000 - Val Loss: 100.7700 - Val KL: 22.9768 - Val Recon: 77.7933
Model saved to /home/benjiy/repo/Verified-Synthetic-Data/MNIST/model_saved/cvae_conv_disc_conv_q0.2_200000_on_conv_q0.2_200000.pth
Model Name: cvae_conv_disc_conv_q0.2_500000_on_conv_q0.2_200000 - Val Loss: 100.5748 - Val KL: 23.1116 - Val Recon: 77.4632
Model saved to /home/benjiy/repo/Verified-Synthetic-Data/MNIST/model_saved/cvae_conv_disc_conv_q0.2_500000_on_conv_q0.2_200000.pth
Model Name: cvae_conv_disc_conv_q0.2_1000000_on_conv_q0.2_200000 - Val Loss: 100.3117 - Val KL: 23.1086 - Val Recon: 77.2031
Model saved to /home/benjiy/repo/Verified-Synthetic-Data/MNIST/model_saved/cvae_conv_disc_conv_q0.2_1000000_on_conv_q0.2_200000.pth
Model Name: cvae_conv_disc_conv_q0.8_200000_on_conv_q0.2_200000 - Val Loss: 101.3136 - Val KL: 22.8376 - Val Recon: 78.4760
Model saved to /home/benjiy/repo/Verified-Synthetic-Data/MNIST/model_saved/cvae_conv_disc_conv_q0.8_200000_on

In [15]:
res_table.to_csv(os.path.join(results_saved_path, 'disc_conv_onestep_q_combine_size_results.csv'),index=False,header=False,mode='a')

In [14]:
res_table

,val_loss,val_recon,val_kl,model_name
0,100.334020,77.888820,22.445201,conv_conv_real_5000
1,103.695988,81.281328,22.414660,cvae_conv_disc_mlp_q0.01_200000_on_real_5000
2,99.654938,77.716163,21.938775,cvae_conv_disc_mlp_q0.05_200000_on_real_5000
3,98.963428,77.335351,21.628077,cvae_conv_disc_mlp_q0.1_200000_on_real_5000
4,98.656953,76.866568,21.790386,cvae_conv_disc_mlp_q0.2_200000_on_real_5000
5,98.676513,76.941153,21.735361,cvae_conv_disc_mlp_q0.5_200000_on_real_5000
